# 06C 从记录判断发生了什么

现在我们有了持续任务。最后要学会区分恢复状态、诊断事件、原始产物和最终答案。它们各回答不同问题。

本份 Notebook 包含实际跨进程 DeepSeek 记录。默认读取保存结果，不发新模型请求。

**本实践在总览中的位置**：把前两份 Notebook 的状态与恢复结论放回完整运行证据中，沿事件、产物和源码快照判断“任务是否完成、结论是否可信”。

**贯穿案例**

读取一条已保存的真实源码导读运行：第一进程退出，第二进程接上，最终回答列出了配置相关文件。检查点说明任务能否继续，JSONL 事件说明途中发生了什么，产物引用让我们回读长工具输出；最终仍需逐行核对源码，不能把 `final_answer` 当作事实验收。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：诊断日志怎样带你找到原始产物；把真实跨进程 Trace 还原成时间线。

课件页：L06.03-S03、L06.06-S01、L06.06-S02。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, asyncio, tempfile, copy, inspect, subprocess, hashlib
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "06 Harness 与持续任务运行"
sys.path.insert(0, str(CHAPTER / "代码"))
from harness_lab import Harness, SimulatedCrash, source_snapshot
from checkpoint_store import CheckpointStore, CheckpointConflict, encode
from runtime_events import EventLog
from agent_lab import ReplayModel, tool_response
from skill_lab import SkillLibrary
from IPython.display import display, Code
print("随课只读工具；SQLite 保存任务；JSONL 保存诊断事件。")


随课只读工具；SQLite 保存任务；JSONL 保存诊断事件。


## 1. 事件日志记录事实，不承担恢复

trace_id 连接同一个任务的事件，span_id 标识具体步骤。事件名说明哪一步开始、完成或失败；metrics 保存数值；payload 只放内容引用与大小。

SQLite 检查点是恢复依据，JSONL 是诊断材料。如果事件日志写入失败，任务仍可继续；如果检查点保存失败，则不能假装已经保存后继续执行。


In [2]:
display(Code(inspect.getsource(EventLog), language="python"))


class EventLog:
    def __init__(self,path,trace_id):
        self.path,self.trace_id=Path(path),trace_id
        self.dropped=0

    def emit(self,event_name,status,span_id=None,attributes=None,metrics=None,payload=None,error=None):
        span_id=span_id or uuid.uuid4().hex[:16]
        event={'schemaVersion':'1','timestamp':datetime.now(timezone.utc).isoformat(),
            'eventName':event_name,'stage':event_name.rsplit('.',1)[-1],
            'status':status,'severity':'error' if status=='failed' else 'info',
            'trace':{'traceId':self.trace_id,'spanId':span_id,'parentSpanId':self.trace_id[:16],
                     'traceparent':f'00-{self.trace_id}-{span_id}-01'},
            'attributes':attributes or {},'metrics':metrics or {}}
        if span_id==self.trace_id[:16]:event['trace'].pop('parentSpanId')
        if payload is not None:event['payload']={'contentMode':'metadata_only',**payload}
        if error is not None:event['error']=error
        # 诊断写入失败不改变任务结果；检查点失败则必须阻止继续执行。
        try:
            self.path.parent.mkdir(parents=True,exist_ok=True)
            with self.path.open('a',encoding='utf-8') as f:f.write(json.dumps(event,ensure_ascii=False)+'\n')
        except (OSError,ValueError,TypeError):self.dropped+=1
        return span_id

In [3]:
record = json.loads((CHAPTER / "数据/实验记录/deepseek-harness.json").read_text(encoding='utf-8'))
events = record["events"]
for event in events:
    if event["eventName"] in {"course.model.started", "course.model.completed", "course.tool.completed", "course.task.status_changed"}:
        print(event["eventName"], event["status"], event["attributes"], event["metrics"])
trace_ids = {e["trace"]["traceId"] for e in events}
print("跨进程仍属于同一 trace:", len(trace_ids) == 1)


course.model.started started {} {'request_ordinal': 1}
course.model.completed succeeded {} {'duration_ms': 1266.379}
course.tool.completed succeeded {'tool_name': 'activate_skill'} {'duration_ms': 4.404}
course.tool.completed succeeded {'tool_name': 'mcp_list_files'} {'duration_ms': 8.668}
course.model.started started {} {'request_ordinal': 2}
course.model.completed succeeded {} {'duration_ms': 1349.112}
course.tool.completed succeeded {'tool_name': 'mcp_read_file'} {'duration_ms': 7.839}
course.tool.completed succeeded {'tool_name': 'mcp_read_file'} {'duration_ms': 7.199}
course.tool.completed succeeded {'tool_name': 'mcp_read_file'} {'duration_ms': 8.852}
course.tool.completed succeeded {'tool_name': 'mcp_read_file'} {'duration_ms': 8.153}
course.model.started started {} {'request_ordinal': 3}
course.model.completed succeeded {} {'duration_ms': 4536.241}
course.task.status_changed succeeded {'task_status': 'final_answer'} {}
跨进程仍属于同一 trace: True


### 把真实跨进程 Trace 还原成时间线

配套课件：L06.03-S03、L06.06-S01、L06.06-S02。

上一格读取的是本次真实模型任务产生的事件。用 pandas 解析时间戳并按阶段汇总，检查两个进程是否共享运行 ID；事件统计用于定位，不把日志当成最终答案或可恢复检查点。

In [4]:
import pandas as pd
case_events = pd.DataFrame([{"time": e["timestamp"], "event": e["eventName"],
    "stage": e["stage"], "status": e["status"],
    "trace_id": e["trace"]["traceId"]} for e in events])
case_events["time"] = pd.to_datetime(case_events["time"], utc=True)
case_events = case_events.sort_values("time")
print(case_events.groupby(["stage", "status"]).size().to_string())
print("首尾事件:", case_events.iloc[0]["event"], "→", case_events.iloc[-1]["event"])
assert case_events["trace_id"].nunique() == 1
assert record["first_process"]["run_id"] == record["second_process"]["run_id"]
print("同一任务跨进程继续；仍需对照检查点修订号和产物哈希。")


stage           status   
completed       succeeded    36
started         started      10
status_changed  succeeded     1
首尾事件: course.agent.started → course.task.status_changed
同一任务跨进程继续；仍需对照检查点修订号和产物哈希。


**结果解读**

Trace 把发生顺序、阶段和状态串起来；相同 trace/run ID 能关联两段进程记录，不能单独证明副作用只执行一次。需要继续核对检查点与产物。

**小练习**

如果日志丢了一条 tool.completed，但检查点已有产物，恢复时应相信哪一层记录？

<details><summary>完成后展开参考分析</summary>

恢复决策以已提交检查点和幂等/产物证据为准；日志只帮助诊断。缺日志应标记观测缺口，不能仅凭日志缺失重复执行副作用。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

## 2. 大内容按引用核查

工具事件里不复制全部源码，而是记录产物 ID、哈希和字节数。需要看原文时，再从同一任务的产物库或课程保存记录读取。

这减少日志重复，但并不意味着产物没有内容敏感性。当前记录只含课程自编资料；换成个人数据时应改变保存和分享范围。


In [5]:
tool_event = next(e for e in events if e["eventName"] == "course.tool.completed")
print("事件中的 payload:", tool_event["payload"])
artifact_id = tool_event["payload"]["id"]
artifact = next(a for a in record["artifacts"] if a["id"] == artifact_id)
print("随课正文种类:", list(artifact["body"]))
assert hashlib.sha256(encode(artifact["body"]).encode()).hexdigest() == artifact["sha256"]
print("哈希相符；它证明内容一致，不能证明模型结论正确。")


事件中的 payload: {'contentMode': 'metadata_only', 'id': '00c16194935144885b97c99bc84096c6c9e93c1b37f2d0cb01cf0c8cd4d5867f', 'size_bytes': 992, 'sha256': '00c16194935144885b97c99bc84096c6c9e93c1b37f2d0cb01cf0c8cd4d5867f'}
随课正文种类: ['instructions', 'ok']
哈希相符；它证明内容一致，不能证明模型结论正确。


### 诊断日志怎样带你找到原始产物

配套课件：L06.06-S01、L06.06-S02。

选择一个真实保存记录中的产物，按事件 payload.id 找到正文，再核对 SHA-256。随后只修改副本中的正文，观察指纹失配。这个检查回答“正文是否被改动”，不回答“模型说得是否正确”。

In [6]:
case_body = copy.deepcopy(artifact["body"])
case_original_hash = hashlib.sha256(encode(case_body).encode()).hexdigest()
case_body["classroom_changed"] = True  # 只改内存副本，保留原始证据
case_changed_hash = hashlib.sha256(encode(case_body).encode()).hexdigest()
print("事件引用:", artifact_id)
print("原正文一致:", case_original_hash == artifact["sha256"])
print("改动后仍一致:", case_changed_hash == artifact["sha256"])
assert case_original_hash == artifact["sha256"] and case_changed_hash != artifact["sha256"]


事件引用: 00c16194935144885b97c99bc84096c6c9e93c1b37f2d0cb01cf0c8cd4d5867f
原正文一致: True
改动后仍一致: False


**结果解读**

原正文指纹相符，改动后不符。事件负责定位，产物保存完整内容，检查点负责继续执行；三种记录各有职责。哈希匹配后仍应回到源码核验答案。

**小练习**

如果日志缺一条，但检查点和产物完整，能不能仅凭日志判断任务从未执行？

<details><summary>完成后展开参考分析</summary>

不能。日志可能丢失；需要结合状态与产物。反过来只有日志成功事件，也不足以证明检查点已经保存。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

## 3. 两个真实进程，共同推进一次模型任务

第一次启动请求 DeepSeek，保存提出的工具动作后退出，此时还未执行工具。第二个进程加载同一检查点，执行已保存动作并继续后续请求。

这证明了本实现能跨进程继续此任务，不证明所有中断位置都能无损恢复。06B 的不确定窗口反例仍然成立。


In [7]:
first, second = record["first_process"], record["second_process"]
state = second["state"]
print("第一进程退出时:", first["state"]["phase"], "工具次数:", first["state"]["tool_calls"])
print("第二进程结束时:", state["status"], "请求次数:", state["turns"], "工具次数:", state["tool_calls"])
assert first["run_id"] == second["run_id"]
print("实际用量:", sum(u["total_tokens"] for u in state["usage"]))
print(state["answer"])


第一进程退出时: tools_ready 工具次数: 0
第二进程结束时: final_answer 请求次数: 3 工具次数: 6
实际用量: 5946
**调用关系（一句话）：** `main.py:11` → `main()` (`main.py:5`) → `from config import load_settings` (`main.py:2`) → `load_settings()` (`config.py:7`) → 返回字典 → `main.py:7` 打印欢迎语。

**1. 入口如何进入 main**

- `main.py:10` 用 `if __name__ == "__main__":` 判断「本文件是否被直接运行」。
- 成立时才调用 `main()`（`main.py:11`）。注意：这只是源码里写的判断，我无法确认它是否真的被执行过。

**2. main 如何调用配置加载**

- `main.py:2` 导入 `load_settings`（模块 `config`，未用包前缀，说明假设 `config.py` 与 `main.py` 在同一目录/同一导入路径下）。
- `main.py:6` 调用 `load_settings()`，把返回值存进 `settings`。
- `main.py:7` 用 `settings['course']` 拼出欢迎语。这里隐含一个条件：返回的字典**必须**含 `course` 键，否则会 KeyError。

**3. 配置路径的分支（config.py:7–11）**

- `config.py:8` 是第一处分支点：`os.environ.get("STUDYBOX_CONFIG", "settings.json")`。
  - 若环境变量 `STUDYBOX_CONFIG` 已设置 → 用它的值做路径；
  - 未设置 → 退回默认值 `"settings.json"`（相对路径）。
- `config.py:9` 第二处分支：`if not path.exists():`
  - **缺失分支**（`config.py:9-10`）：返回硬编码兜底字典 `{"course": "大模型应用开发", "timeout": 30}`。
  - **存在分支**（`config.py:

## 4. 恢复成功之后，还要检查答案

最终文本表示本轮停止。读原文时依次核对：入口与调用行号；默认路径与环境变量；相对路径的工作目录条件；缺失分支；JSON 顶层与字段是否真的经过校验。

回答如果笼统说“总是返回含 course 的字典”，就超出了代码保证。即便后面又说没有字段校验，开头的错误保证仍应该指出。Harness 管理运行，不自动修正事实。


In [8]:
snapshot_checks = {
    relative: hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected_hash
    for relative, expected_hash in state["source_snapshot"].items()
}
unchanged_checks = {path: ok for path, ok in snapshot_checks.items()
                    if path not in record["adapted_student_files"]}
assert unchanged_checks and all(unchanged_checks.values())
print("未改动示例文件的历史哈希核验通过。")
print("当前自编资料与本次运行快照一致。")
for path in ["main.py", "config.py"]:
    text = (ROOT / "02 从大模型到 Agent 组成与最小循环/数据/demo-repo" / path).read_text(encoding='utf-8')
    for i, line in enumerate(text.splitlines(),1): print(f"{path}:{i} {line}")


未改动示例文件的历史哈希核验通过。
当前自编资料与本次运行快照一致。
main.py:1 """StudyBox 运行入口。"""
main.py:2 from config import load_settings
main.py:3 
main.py:4 
main.py:5 def main():
main.py:6     settings = load_settings()
main.py:7     print(f"欢迎，{settings['course']}！")
main.py:8 
main.py:9 
main.py:10 if __name__ == "__main__":
main.py:11     main()
config.py:1 """配置读取；本仓库只作为 Agent 阅读资料。"""
config.py:2 import json
config.py:3 import os
config.py:4 from pathlib import Path
config.py:5 
config.py:6 
config.py:7 def load_settings():
config.py:8     path = Path(os.environ.get("STUDYBOX_CONFIG", "settings.json"))
config.py:9     if not path.exists():
config.py:10         return {"course": "大模型应用开发", "timeout": 30}
config.py:11     return json.loads(path.read_text(encoding="utf-8"))


## 练习：日志故障与状态故障

让日志路径的父目录变成一个普通文件，运行一次 emit。预测会不会抛异常，以及 dropped 如何变化。再解释：为什么检查点写入失败不能使用同样的“忽略后继续”策略？


In [9]:
with tempfile.TemporaryDirectory() as d:
    blocker = Path(d) / "not-a-directory"
    blocker.write_text("课堂障碍文件", encoding='utf-8')
    log = EventLog(blocker / "events.jsonl", "1" * 32)
    log.emit("course.demo.started", "started")
    print("丢弃的诊断事件:", log.dropped)


丢弃的诊断事件: 1


In [10]:
assert log.dropped == 1
print("日志记录失败没有改变业务动作；但检查点若未保存，恢复者就无法据此继续。")


日志记录失败没有改变业务动作；但检查点若未保存，恢复者就无法据此继续。


## 可带走的结果：一次可诊断运行报告

把保存记录中的模型、跨进程阶段、事件数量、产物校验和事实审阅结论整理成 JSON 与 Markdown。报告保留的是这次历史运行使用的模型名；未来重新运行前，应先列出当前账号可用模型，再选择仍受支持的版本。


In [11]:
report_dir = ROOT / ".local/student-runs/l06"
report_dir.mkdir(parents=True, exist_ok=True)
event_counts = {}
for event in events:
    event_counts[event["eventName"]] = event_counts.get(event["eventName"], 0) + 1
report = {
    "report_type": "diagnostic-run-report",
    "recorded_at": record["recorded_at"],
    "run_model": record["model"],
    "model_selection_note": "Before a live rerun, list models available to the current account and choose a supported one.",
    "run_id": first["run_id"],
    "cross_process": {
        "first_phase": first["state"]["phase"],
        "final_status": state["status"],
        "same_run_id": first["run_id"] == second["run_id"],
        "single_trace": len(trace_ids) == 1,
    },
    "usage": {
        "model_attempts": state["turns"],
        "tool_attempts": state["tool_calls"],
        "total_tokens": sum(item["total_tokens"] for item in state["usage"]),
    },
    "evidence": {
        "event_counts": event_counts,
        "artifact_id": artifact_id,
        "artifact_sha256_verified": hashlib.sha256(encode(artifact["body"]).encode()).hexdigest() == artifact["sha256"],
        "unchanged_historical_sources_verified": all(unchanged_checks.values()),
        "student_skill_adapted": record["adapted_student_files"],
        "dropped_demo_events": log.dropped,
    },
    "review": {
        "runtime_recovery_succeeded": state["status"] == "completed",
        "factual_correctness_requires_source_review": True,
        "human_review_required_for_uncertain_side_effects": True,
    },
}
json_path = report_dir / "diagnostic-run-report.json"
md_path = report_dir / "diagnostic-run-report.md"
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
md_path.write_text(
    "# 一次可诊断运行报告\n\n"
    f"- 本次模型：`{report['run_model']}`\n"
    f"- 运行 ID：`{report['run_id']}`\n"
    f"- 跨进程恢复：{'通过' if report['cross_process']['same_run_id'] else '未通过'}\n"
    f"- 最终状态：`{report['cross_process']['final_status']}`\n"
    f"- 产物哈希：{'通过' if report['evidence']['artifact_sha256_verified'] else '未通过'}\n"
    f"- 未改动源码快照：{'通过' if report['evidence']['unchanged_historical_sources_verified'] else '未通过'}\n\n"
    "> 运行恢复成功不等于答案事实正确；重新联网运行前先查询当前账号可用模型。\n",
    encoding="utf-8",
)
print("已保存诊断报告:", json_path.relative_to(ROOT), "和", md_path.relative_to(ROOT))
print(json.dumps(report, ensure_ascii=False, indent=2))


已保存诊断报告: .local\student-runs\l06\diagnostic-run-report.json 和 .local\student-runs\l06\diagnostic-run-report.md
{
  "report_type": "diagnostic-run-report",
  "recorded_at": "2026-09-24T05:47:52.502108+00:00",
  "run_model": "deepseek-flash",
  "model_selection_note": "Before a live rerun, list models available to the current account and choose a supported one.",
  "run_id": "298bbd83cb50466c89910170e0bd7679",
  "cross_process": {
    "first_phase": "tools_ready",
    "final_status": "final_answer",
    "same_run_id": true,
    "single_trace": true
  },
  "usage": {
    "model_attempts": 3,
    "tool_attempts": 6,
    "total_tokens": 5946
  },
  "evidence": {
    "event_counts": {
      "course.agent.started": 1,
      "course.context.completed": 3,
      "course.checkpoint.completed": 18,
      "course.model.started": 3,
      "course.model.completed": 3,
      "course.permission.completed": 6,
      "course.tool.started": 6,
      "course.tool.completed": 6,
      "course.task.status_c

## 本节总结

任务通过明确状态继续，产物通过引用回查，事件通过同一 trace 关联。中断后先判断动作是否确定完成，预算与权限由运行时代码执行。

下一节独立学习智能体评估：怎样定义任务成功、收集多次试验、比较一次改动，并分析实际错误。

重新请求模型使用课程单元 README 的显式命令；Notebook 默认保持离线。
